# Cell 1: Imports & configuration

In [1]:
from pathlib import Path
import numpy as np
import xarray as xr
from tqdm import tqdm

INPUT_DIR = Path("../../data/raw/west_bob")
OUTPUT_DIR = Path("../../data/interim/west_bob")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Global configuration ----
LAT_MODULUS = 20   # grid/modulus for latitude
LON_MODULUS = 20   # grid/modulus for longitude
PREFER_SIDE = "right"  # "left" or "right" — which side gets the extra unit if padding is odd


LAT_DIRECTION = "nearest"   # "add", "subtract", or "nearest"
LON_DIRECTION = "nearest"       # example: different choice for lon
PAD_TIE = "subtract"             # applies when a direction is "nearest"

# For reproducibility in logging
xr.set_options(display_style="text")


# Cell 2: Helpers to detect lat/lon and compute padding

In [2]:
from typing import Dict, Tuple, Literal

def _choose_name(ds, candidates):
    """Return the first matching name from candidates present in ds.coords or ds.data_vars."""
    for name in candidates:
        if name in ds.coords or name in ds.data_vars:
            return name
    return None

def detect_lat_lon(ds) -> Dict[str, object]:
    """
    Detect latitude and longitude coordinate/variable names and their dims.
    Supports 1D coords (e.g., lat(dim='lat'), lon(dim='lon'))
    and 2D coords (e.g., lat(dim=('y','x')), lon(dim=('y','x'))).

    Returns:
        dict with keys: lat_name, lon_name, lat_dims, lon_dims, lat_dim, lon_dim
    """
    # Add names commonly found in SWOT L3 products
    lon_candidates = ["lon", "longitude", "x", "rlon", "XLONG", "nav_lon", "grid_longitude"]
    lat_candidates = ["lat", "latitude", "y", "rlat", "XLAT", "nav_lat", "grid_latitude"]

    # Typical SWOT dims
    lon_dim_candidates = ["lon", "x", "rlon", "num_pixels", "i", "easting"]
    lat_dim_candidates = ["lat", "y", "rlat", "num_lines", "j", "northing"]

    lon_name = _choose_name(ds, lon_candidates)
    lat_name = _choose_name(ds, lat_candidates)

    info = {
        "lon_name": lon_name,
        "lat_name": lat_name,
        "lon_dims": None,
        "lat_dims": None,
        "lon_dim": None,
        "lat_dim": None,
    }

    # If found as variables/coords, use their dims
    if lon_name is not None:
        info["lon_dims"] = tuple(ds[lon_name].dims)
        if len(info["lon_dims"]) == 1:
            info["lon_dim"] = info["lon_dims"][0]
    if lat_name is not None:
        info["lat_dims"] = tuple(ds[lat_name].dims)
        if len(info["lat_dims"]) == 1:
            info["lat_dim"] = info["lat_dims"][0]

    # If not found by variable names, try matching dims directly
    if info["lon_dim"] is None:
        for d in lon_dim_candidates:
            if d in ds.dims:
                info["lon_dim"] = d
                break
    if info["lat_dim"] is None:
        for d in lat_dim_candidates:
            if d in ds.dims:
                info["lat_dim"] = d
                break

    # Handle 2D lat/lon where both exist and share dims (e.g., ('num_lines','num_pixels'))
    if info["lon_dims"] and info["lat_dims"] and len(info["lon_dims"]) == 2 and info["lon_dims"] == info["lat_dims"]:
        info["lat_dim"] = info["lat_dims"][0]
        info["lon_dim"] = info["lat_dims"][1]

    if info["lat_dim"] is None or info["lon_dim"] is None:
        raise ValueError(
            "Unable to detect latitude/longitude dimensions. "
            "Please ensure your dataset has recognizable coord/var names "
            "(e.g., lat/longitude, XLAT/XLONG, nav_lat/nav_lon, num_lines/num_pixels, y/x)."
        )
    return info


def compute_pad(n, modulus=10, prefer="right", direction="add", tie="add"):
    """
    Compute minimal padding to make `n` divisible by `modulus`.

    Returns (left_amount, right_amount, total_amount), where amounts are
    non-negative integers. Interpret them as "units to add" when direction
    is "add" (pad up) or "units to remove" when direction is "subtract"
    (pad down).

    Parameters
    ----------
    n : int
        Current size/count.
    modulus : int, default 10
        Target divisor; must be > 0.
    prefer : {"left","right"}, default "right"
        If the total change is odd, the extra 1 goes to this side.
    direction : {"add","subtract","nearest"}, default "add"
        - "add": compute how much to add to reach the next multiple (ceil).
        - "subtract": compute how much to remove to reach the previous multiple (floor).
        - "nearest": choose whichever requires the smaller total change.
    tie : {"add","subtract"}, default "add"
        Used only when direction="nearest" and add/remove amounts are equal;
        decides which way to go.

    Notes
    -----
    - Works for negative `n` as well, using Python's modulo semantics.
    - If `n` is already divisible by `modulus`, returns (0, 0, 0).
    """
    if modulus <= 0:
        raise ValueError("modulus must be a positive integer")

    # Amount to ADD to reach the next multiple (non-negative)
    add_total = (-n) % modulus
    # Amount to SUBTRACT to reach the previous multiple (non-negative)
    sub_total = n % modulus

    # Choose the operation
    if direction == "add":
        total = add_total
        op = "add"
    elif direction == "subtract":
        total = sub_total
        op = "subtract"
    elif direction == "nearest":
        if add_total < sub_total:
            total = add_total
            op = "add"
        elif sub_total < add_total:
            total = sub_total
            op = "subtract"
        else:  # equal distance to both sides
            op = tie  # "add" or "subtract"
            total = add_total if tie == "add" else sub_total
    else:
        raise ValueError("direction must be 'add', 'subtract', or 'nearest'")

    # Split the total change between left/right as evenly as possible
    left = total // 2
    right = total - left

    # If odd, give the extra unit to the preferred side
    if total % 2 == 1 and prefer == "left":
        left, right = right, left

    return left, right, total


def ensure_float_for_nan(da):
    """If DataArray dtype is integer/unsigned integer, cast to float32 to support NaN padding."""
    if np.issubdtype(da.dtype, np.integer) or np.issubdtype(da.dtype, np.unsignedinteger):
        return da.astype(np.float32)
    return da

# Cell 3: Main padding routine

In [3]:
def pad_dataset_to_multiple_of_x(ds):
    """
    Make ds sizes along latitude and longitude divisible by LAT_MODULUS/LON_MODULUS
    by either padding with NaN (add) or trimming (subtract), according to the
    per-axis global directions LAT_DIRECTION and LON_DIRECTION (supports 'nearest' via tie).
    Returns (ds_out, summary).
    """
    info = detect_lat_lon(ds)
    lat_dim = info["lat_dim"]
    lon_dim = info["lon_dim"]

    # Current sizes
    n_lat = ds.sizes[lat_dim]
    n_lon = ds.sizes[lon_dim]

    # Resolve 'nearest' into a concrete direction per axis (no change to compute_pad)
    def _resolve_direction(n, modulus, direction, tie):
        if direction in ("add", "subtract"):
            return direction
        # direction == "nearest"
        add_total = (-n) % modulus
        sub_total = n % modulus
        if add_total < sub_total:
            return "add"
        elif sub_total < add_total:
            return "subtract"
        else:
            return tie if tie in ("add", "subtract") else "add"

    lat_dir = _resolve_direction(n_lat, LAT_MODULUS, LAT_DIRECTION, PAD_TIE)
    lon_dir = _resolve_direction(n_lon, LON_MODULUS, LON_DIRECTION, PAD_TIE)

    # Use your existing compute_pad to get left/right + total counts
    lat_left, lat_right, lat_change = compute_pad(
        n_lat, LAT_MODULUS, PREFER_SIDE, direction=lat_dir, tie=PAD_TIE
    )
    lon_left, lon_right, lon_change = compute_pad(
        n_lon, LON_MODULUS, PREFER_SIDE, direction=lon_dir, tie=PAD_TIE
    )

    # Compute after sizes based on chosen direction
    n_lat_after = n_lat + lat_change if lat_dir == "add" else n_lat - lat_change
    n_lon_after = n_lon + lon_change if lon_dir == "add" else n_lon - lon_change

    summary = {
        "lat_dim": lat_dim,
        "lon_dim": lon_dim,
        "n_lat_before": n_lat,
        "n_lon_before": n_lon,
        "lat_operation": lat_dir,               # "add" or "subtract"
        "lon_operation": lon_dir,
        "lat_pad": (lat_left, lat_right),       # counts applied to left/right
        "lon_pad": (lon_left, lon_right),
        "n_lat_after": n_lat_after,
        "n_lon_after": n_lon_after,
        "lat_modulus": LAT_MODULUS,
        "lon_modulus": LON_MODULUS,
    }

    # If no change needed, return original dataset
    if lat_change == 0 and lon_change == 0:
        return ds, summary

    # 1) Trim first (for axes where operation is 'subtract')
    indexer = {}
    if lat_dir == "subtract" and lat_change > 0:
        indexer[lat_dim] = slice(lat_left, n_lat - lat_right)
    if lon_dir == "subtract" and lon_change > 0:
        indexer[lon_dim] = slice(lon_left, n_lon - lon_right)

    ds_work = ds.isel(**indexer) if indexer else ds

    # 2) Then pad (for axes where operation is 'add')
    pad_map = {}
    if lat_dir == "add" and lat_change > 0:
        pad_map[lat_dim] = (lat_left, lat_right)
    if lon_dir == "add" and lon_change > 0:
        pad_map[lon_dim] = (lon_left, lon_right)

    if not pad_map:
        # Only trimming was required
        return ds_work, summary

    # Prepare padded coords & variables (avoid in-place mutation of ds_work)
    new_coords = {}
    for coord_name, da in ds_work.coords.items():
        if (lat_dim in da.dims) or (lon_dim in da.dims):
            da2 = ensure_float_for_nan(da)
            da2 = da2.pad(pad_map, mode="constant", constant_values=np.nan)
            new_coords[coord_name] = da2
        else:
            new_coords[coord_name] = da

    new_vars = {}
    for var_name, da in ds_work.data_vars.items():
        if (lat_dim in da.dims) or (lon_dim in da.dims):
            da2 = ensure_float_for_nan(da)
            da2 = da2.pad(pad_map, mode="constant", constant_values=np.nan)
            da2.attrs = da.attrs  # preserve attrs
            new_vars[var_name] = da2
        else:
            new_vars[var_name] = da

    ds_out = xr.Dataset(new_vars, coords=new_coords, attrs=ds_work.attrs)
    return ds_out, summary

def process_all_files(input_dir=INPUT_DIR, output_dir=OUTPUT_DIR):
    """
    Iterate through all .nc files, pad them if needed, and save to the output directory.
    Returns a list of summaries for reporting.
    """
    files = sorted(input_dir.glob("*.nc"))
    if not files:
        print(f"No .nc files found in {input_dir}")
        return []

    summaries = []
    for fp in tqdm(files, desc="Padding NetCDF files", unit="file"):
        ds = xr.open_dataset(fp)  # lazy by default

        try:
            ds_padded, summary = pad_dataset_to_multiple_of_x(ds)
        except Exception as e:
            print(f"[ERROR] {fp.name}: {e}")
            ds.close()
            continue

        out_fp = output_dir / fp.name
        ds_padded.to_netcdf(out_fp)

        ds.close()
        ds_padded.close()

        summary.update({"file": fp.name, "output_path": str(out_fp)})
        summaries.append(summary)

    return summaries

# Cell 4: Execute and print a concise report

In [4]:
summaries = process_all_files()

print("\n=== Padding Summary ===")
for s in summaries:
    print(
        f"{s['file']}: "
        f"{s['lat_dim']} {s['n_lat_before']} -> {s['n_lat_after']} (pad {s['lat_pad']}), "
        f"{s['lon_dim']} {s['n_lon_before']} -> {s['n_lon_after']} (pad {s['lon_pad']}) "
        f"-> {s['output_path']}"
    )



Padding NetCDF files:   0%|          | 0/961 [00:00<?, ?file/s]/var/folders/d4/f03ry9712tx6k88b22dqnwmm0000gq/T/ipykernel_81472/3150169069.py:1: SerializationWarning: saving variable latitude with floating point data as an integer dtype without any _FillValue to use for NaNs
  summaries = process_all_files()
/var/folders/d4/f03ry9712tx6k88b22dqnwmm0000gq/T/ipykernel_81472/3150169069.py:1: SerializationWarning: saving variable longitude with floating point data as an integer dtype without any _FillValue to use for NaNs
  summaries = process_all_files()
Padding NetCDF files:   0%|          | 1/961 [00:01<19:37,  1.23s/file]/var/folders/d4/f03ry9712tx6k88b22dqnwmm0000gq/T/ipykernel_81472/3150169069.py:1: SerializationWarning: saving variable latitude with floating point data as an integer dtype without any _FillValue to use for NaNs
  summaries = process_all_files()
/var/folders/d4/f03ry9712tx6k88b22dqnwmm0000gq/T/ipykernel_81472/3150169069.py:1: SerializationWarning: saving variable long


=== Padding Summary ===
SWOT_L3_LR_SSH_Expert_001_161_20230726T224518_20230726T233644_v3.0_with_internal_wave_bbox_mask.nc: num_lines 709 -> 700 (pad (4, 5)), num_pixels 69 -> 60 (pad (4, 5)) -> ../../data/interim/west_bob/SWOT_L3_LR_SSH_Expert_001_161_20230726T224518_20230726T233644_v3.0_with_internal_wave_bbox_mask.nc
SWOT_L3_LR_SSH_Expert_001_189_20230727T224549_20230727T233716_v3.0_with_internal_wave_bbox_mask.nc: num_lines 709 -> 700 (pad (4, 5)), num_pixels 69 -> 60 (pad (4, 5)) -> ../../data/interim/west_bob/SWOT_L3_LR_SSH_Expert_001_189_20230727T224549_20230727T233716_v3.0_with_internal_wave_bbox_mask.nc
SWOT_L3_LR_SSH_Expert_001_217_20230728T224620_20230728T233747_v3.0_with_internal_wave_bbox_mask.nc: num_lines 709 -> 700 (pad (4, 5)), num_pixels 69 -> 60 (pad (4, 5)) -> ../../data/interim/west_bob/SWOT_L3_LR_SSH_Expert_001_217_20230728T224620_20230728T233747_v3.0_with_internal_wave_bbox_mask.nc
SWOT_L3_LR_SSH_Expert_001_245_20230729T224652_20230729T233818_v3.0_with_internal_

# Cell 5 (Optional): Synthetic demo

In [6]:
ny, nx = 68, 706  # like your example
lat = xr.DataArray(np.linspace(0, 1, ny), dims=("lat",), name="lat")
lon = xr.DataArray(np.linspace(0, 1, nx), dims=("lon",), name="lon")
z = xr.DataArray(np.random.rand(ny, nx), dims=("lat", "lon"), name="z")

ds_demo = xr.Dataset({"z": z}, coords={"lat": lat, "lon": lon})
ds_padded, summary = pad_dataset_to_multiple_of_x(ds_demo)

print(summary)
print("z shape before:", ds_demo["z"].shape, "after:", ds_padded["z"].shape)
print("lat length before:", ds_demo.dims["lat"], "after:", ds_padded.dims["lat"])
print("lon length before:", ds_demo.dims["lon"], "after:", ds_padded.dims["lon"])

{'lat_dim': 'lat', 'lon_dim': 'lon', 'n_lat_before': 68, 'n_lon_before': 706, 'lat_operation': 'subtract', 'lon_operation': 'subtract', 'lat_pad': (4, 4), 'lon_pad': (3, 3), 'n_lat_after': 60, 'n_lon_after': 700, 'lat_modulus': 20, 'lon_modulus': 20}
z shape before: (68, 706) after: (60, 700)
lat length before: 68 after: 60
lon length before: 706 after: 700


/var/folders/d4/f03ry9712tx6k88b22dqnwmm0000gq/T/ipykernel_81472/640496231.py:11: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("lat length before:", ds_demo.dims["lat"], "after:", ds_padded.dims["lat"])
/var/folders/d4/f03ry9712tx6k88b22dqnwmm0000gq/T/ipykernel_81472/640496231.py:12: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("lon length before:", ds_demo.dims["lon"], "after:", ds_padded.dims["lon"])


In [1]:
# Cell: Load first padded file and print its dimensions
from pathlib import Path
import xarray as xr

OUTPUT_DIR = Path("../../data/interim/west_bob")

# Find the first .nc file (sorted lexicographically)
files = sorted(OUTPUT_DIR.glob("*.nc"))
if not files:
    raise FileNotFoundError(f"No .nc files found in {OUTPUT_DIR}")

first_fp = files[0]
print(f"Loading: {first_fp}")

# Open and inspect
with xr.open_dataset(first_fp) as ds:
    # ds.sizes is the preferred way to get a mapping of dim -> length
    print("\n=== Dimensions (sizes) ===")
    for dim, size in ds.sizes.items():
        print(f"{dim}: {size}")

    # Optional: also show the dims view and a short dataset summary
    print("\n=== dims view (legacy, may change in future) ===")
    print(ds.dims)

    print("\n=== Dataset summary ===")
    ds

Loading: ../../data/interim/west_bob/SWOT_L3_LR_SSH_Expert_001_161_20230726T224518_20230726T233644_v3.0_with_internal_wave_bbox_mask.nc

=== Dimensions (sizes) ===
num_lines: 700
num_pixels: 60
num_nadir: 1384

=== dims view (legacy, may change in future) ===
FrozenMappingWarningOnValuesAccess({'num_lines': 700, 'num_pixels': 60, 'num_nadir': 1384})

=== Dataset summary ===


In [2]:
ds

<xarray.Dataset> Size: 6MB
Dimensions:                           (num_lines: 700, num_pixels: 60,
                                       num_nadir: 1384)
Coordinates:
    latitude                          (num_lines, num_pixels) float64 336kB ...
    longitude                         (num_lines, num_pixels) float64 336kB ...
Dimensions without coordinates: num_lines, num_pixels, num_nadir
Data variables: (12/23)
    time                              (num_lines) datetime64[ns] 6kB ...
    calibration                       (num_lines, num_pixels) float64 336kB ...
    cross_track_distance              (num_pixels) float64 480B ...
    dac                               (num_lines, num_pixels) float64 336kB ...
    internal_tide                     (num_lines, num_pixels) float64 336kB ...
    mdt                               (num_lines, num_pixels) float64 336kB ...
    ...                                ...
    vgosa_filtered                    (num_lines, num_pixels) float64 336kB ...
    vgosa_unfiltered                  (num_lines, num_pixels) float64 336kB ...
    i_num_line                        (num_nadir) int16 3kB ...
    i_num_pixel                       (num_nadir) int8 1kB ...
    internal_wave_bbox_mask           (num_lines, num_pixels) uint8 42kB ...
    ssha_unfiltered_rolling_highpass  (num_lines, num_pixels) float32 168kB ...
Attributes: (12/52)
    Conventions:                     CF-1.9
    Metadata_Conventions:            Unidata Dataset Discovery v1.0
    cdm_data_type:                   Swath
    comment:                         Sea Surface Height measured by Altimetry
    data_used:                       SWOT KaRIn L2_LR_SSH PGC0/PIC0/PIC2/PID0...
    doi:                             https://doi.org/10.24400/527896/A01-2023...
    ...                              ...
    clip_lat_max:                    15.5
    note:                            Clipped only by latitude range. Full cro...
    source_l3_file:                  data/SWOT_L3_Full_Bay_of_Bengal_Dataset/...
    yolo_filter_type:                rolling
    yolo_mask_variable:              internal_wave_bbox_mask
    saved_filtered_variable:         ssha_unfiltered_rolling_highpass